In [1]:
import os
import cv2
import numpy as np
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

2025-04-24 00:12:53.780033: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-04-24 00:12:53.812346: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1745442773.848509   37033 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1745442773.859455   37033 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-04-24 00:12:53.895389: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

In [2]:
FOLDER_PATH = "../../img/ote"
FRAME_COUNT = 16
IMG_SIZE = 64
CLASS_COUNT = 3
class_names = ["0", "1", "2"]

In [3]:
def load_videos(folder, frame_count=FRAME_COUNT, img_size=IMG_SIZE):
    videos = []
    labels = []

    for video_folder in os.listdir(folder):
        video_path = os.path.join(folder, video_folder)
        if os.path.isdir(video_path):
            try:
                _, label = video_folder.split("_")
                label = int(label[0])
            except ValueError:
                print(f"[SKIP] invalid dir name: {video_folder}")
                continue

            frames = []
            frame_files = sorted(os.listdir(video_path))[:frame_count]
            print(f"[INFO] {video_folder} -> {len(frame_files)} frame")

            for filename in frame_files:
                img_path = os.path.join(video_path, filename)
                img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
                if img is not None:
                    img = cv2.resize(img, (img_size, img_size))
                    img = img.flatten()
                    frames.append(img)
                else:
                    print(f"[WARN] frame can not be read!: {img_path}")

            if len(frames) == frame_count:
                videos.append(frames) 
                labels.append(label)
            else:
                print(f"[SKIP] {video_folder} -> not enough frame ({len(frames)}/{frame_count})")

    print(f"[SUMMARY] Sum of videos: {len(videos)}")
    return np.array(videos), np.array(labels)

In [4]:
X, y = load_videos(FOLDER_PATH, FRAME_COUNT, IMG_SIZE)
y = to_categorical(y, num_classes=CLASS_COUNT)

[INFO] 105_100 -> 16 frame
[INFO] 114_000 -> 16 frame
[INFO] 117_100 -> 16 frame
[INFO] 140_000 -> 16 frame
[INFO] 130_000 -> 16 frame
[INFO] 136_000 -> 16 frame
[INFO] 110_000 -> 16 frame
[INFO] 106_111 -> 16 frame
[INFO] 107_000 -> 16 frame
[INFO] 128_200 -> 0 frame
[SKIP] 128_200 -> not enough frame (0/16)
[INFO] 134_000 -> 16 frame
[INFO] 113_000 -> 16 frame
[INFO] 123_000 -> 16 frame
[INFO] 125_100 -> 0 frame
[SKIP] 125_100 -> not enough frame (0/16)
[INFO] 132_000 -> 16 frame
[INFO] 131_101 -> 16 frame
[INFO] 119_010 -> 16 frame
[INFO] 118_000 -> 16 frame
[INFO] 138_100 -> 16 frame
[INFO] 120_000 -> 16 frame
[INFO] 104_000 -> 16 frame
[INFO] 139_110 -> 16 frame
[INFO] 141_000 -> 16 frame
[INFO] 116_000 -> 16 frame
[INFO] 137_000 -> 16 frame
[INFO] 124_000 -> 16 frame
[INFO] 101_000 -> 0 frame
[SKIP] 101_000 -> not enough frame (0/16)
[INFO] 129_100 -> 16 frame
[INFO] 126_100 -> 0 frame
[SKIP] 126_100 -> not enough frame (0/16)
[INFO] 122_000 -> 16 frame
[INFO] 127_000 -> 0 frame


In [5]:
print("Number of videos:", len(X))
print("Shape of data:", X.shape) 

Number of videos: 36
Shape of data: (36, 16, 4096)


In [6]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [7]:
model = Sequential([
    LSTM(128, input_shape=(FRAME_COUNT, IMG_SIZE * IMG_SIZE), return_sequences=False),
    Dropout(0.2),
    Dense(64, activation='relu'),
    Dense(CLASS_COUNT, activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

model.fit(X_train, y_train, epochs=10, batch_size=4, validation_split=0.1)

2025-04-24 00:12:59.571187: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:152] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
/home/tolgahan/Desktop/machine-learning/venv/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 128)            │     2,163,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 3)              │           195 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,171,651 (8.28 MB)

 Trainable params: 2,171,651 (8.28 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10


2025-04-24 00:13:04.347907: E tensorflow/core/util/util.cc:131] oneDNN supports DT_UINT8 only on platforms with AVX-512. Falling back to the default Eigen-based implementation if present.


7/7 ━━━━━━━━━━━━━━━━━━━━ 6s 311ms/step - accuracy: 0.4289 - loss: 1.1292 - val_accuracy: 1.0000 - val_loss: 0.4992
Epoch 2/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 163ms/step - accuracy: 0.7393 - loss: 0.8161 - val_accuracy: 1.0000 - val_loss: 0.3143
Epoch 3/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 122ms/step - accuracy: 0.7486 - loss: 0.7714 - val_accuracy: 1.0000 - val_loss: 0.4208
Epoch 4/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 120ms/step - accuracy: 0.7657 - loss: 0.7892 - val_accuracy: 1.0000 - val_loss: 0.3992
Epoch 5/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 119ms/step - accuracy: 0.6846 - loss: 0.7595 - val_accuracy: 1.0000 - val_loss: 0.4120
Epoch 6/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 131ms/step - accuracy: 0.6861 - loss: 0.7493 - val_accuracy: 1.0000 - val_loss: 0.4123
Epoch 7/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 143ms/step - accuracy: 0.7050 - loss: 0.8746 - val_accuracy: 1.0000 - val_loss: 0.3777
Epoch 8/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 131ms/step - accuracy: 0.7467 - loss: 0.8717 - val_accuracy: 1.0000 - val_loss: 0.3408
Epo

In [8]:
test_loss, test_acc = model.evaluate(X_test, y_test)
print(f"LSTM Model Test Accuracy: {test_acc:.2f}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 124ms/step - accuracy: 0.6250 - loss: 0.7294
LSTM Model Test Accuracy: 0.62
